In [1]:
from __future__  import annotations
import json
import re
from dataclasses  import dataclass
from pathlib import Path

In [2]:
CHUNK_SIZE = 400
CHUNK_OVERLAP = 80
SKIP_UNTIL_PDF_PAGE = 15
MIN_CHARS = 50
_ENCODER = None

In [3]:
@dataclass
class PageSpan:

    pdf_page: int
    printed_page: int | None
    section_ref: str | None
    heading: str | None
    char_start: int
    char_end: int


In [4]:
@dataclass
class TextChunk:

    text: str
    char_start: int
    char_end: int



In [5]:

def _get_encoder():

    try:
        import tiktoken

        return tiktoken.get_encoding("cl100k_base")

    except ImportError:
        return None


In [6]:
def _ensure_encoder():

    global _ENCODER

    if _ENCODER is None:
        _ENCODER = _get_encoder()

    return _ENCODER

In [7]:
def count_tokens(text: str) -> int:

    if not text.strip():
        return 0

    encoder = _ensure_encoder()

    if encoder is not None:
        return len(encoder.encode(text))

    word_count = len(re.findall(r"\S+", text))

    return max(1, int(word_count * 1.25))


In [8]:
def chunk_text(text: str,chunk_size: int = CHUNK_SIZE,overlap: int = CHUNK_OVERLAP,) -> list[TextChunk]:

    if not text.strip():
        return []

    if chunk_size <= 0:
        raise ValueError("chunk_size mora biti veći od 0.")

    if overlap < 0:
        raise ValueError("overlap ne može biti negativan.")

    if overlap >= chunk_size:
        raise ValueError("overlap mora biti manji od chunk_size.")

    encoder = _ensure_encoder()

    if encoder is not None:
        tokens = encoder.encode(text)

        chunks: list[TextChunk] = []
        start_token = 0

        while start_token < len(tokens):
            end_token = min(len(tokens),start_token + chunk_size,)
            text_before_chunk = encoder.decode(tokens[:start_token])
            text_through_chunk = encoder.decode(tokens[:end_token])
            char_start = len(text_before_chunk)
            char_end = len(text_through_chunk)
            chunk_body = encoder.decode(tokens[start_token:end_token])

            chunks.append(TextChunk(text=chunk_body,char_start=char_start,char_end=char_end,))

            if end_token >= len(tokens):
                break

            start_token = end_token - overlap

        return chunks

    estimated_token_count = max(count_tokens(text), 1)
    approx_chars_per_token = len(text) / estimated_token_count
    chunk_chars = max(1,int(chunk_size * approx_chars_per_token),)
    overlap_chars = max(0,int(overlap * approx_chars_per_token),)
    chunks: list[TextChunk] = []
    start_char = 0

    while start_char < len(text):
        end_char = min(len(text),start_char + chunk_chars,)
        chunks.append(TextChunk(text=text[start_char:end_char],char_start=start_char,char_end=end_char,))

        if end_char >= len(text):
            break

        start_char = end_char - overlap_chars

    return chunks

In [9]:
def _pages_for_range(spans: list[PageSpan],start: int,end: int,) -> list[PageSpan]:
    
    return [span for span in spans if span.char_end > start and span.char_start < end]

In [10]:
def _context_prefix(section_ref: str | None,heading: str | None,) -> str:
  
    parts = [part for part in (section_ref, heading) if part]

    if not parts:
        return ""

    return f"[{' > '.join(parts)}]\n\n"



In [11]:
def load_pages(pages_jsonl: Path,) -> tuple[str, list[PageSpan]]:

    if not pages_jsonl.exists():
        raise FileNotFoundError( f"Datoteka nije pronađena: {pages_jsonl.resolve()}")

    document_parts: list[str] = []
    spans: list[PageSpan] = []
    cursor = 0

    with pages_jsonl.open(mode="r",encoding="utf-8",) as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)

            except json.JSONDecodeError as error:
                raise ValueError(f"Neispravan JSON u redu {line_number}.") from error

            pdf_page = record["pdf_page"]
            text = (record.get("text") or "").strip()

            if pdf_page < SKIP_UNTIL_PDF_PAGE:
                continue

            if len(text) < MIN_CHARS:
                continue

            if document_parts:
                document_parts.append("\n\n")
                cursor += 2

            char_start = cursor
            document_parts.append(text)
            cursor += len(text)
            spans.append(PageSpan(pdf_page=pdf_page,printed_page=record.get("printed_page"),section_ref=record.get("section_ref"),heading=record.get("heading"),char_start=char_start,char_end=cursor,))

    document = "".join(document_parts)

    return document, spans

In [12]:
def build_chunks(pages_jsonl: Path,chunk_size: int = CHUNK_SIZE,overlap: int = CHUNK_OVERLAP,) -> list[dict]:

    document, spans = load_pages(pages_jsonl)

    if not document.strip():
        return []

    raw_chunks = chunk_text(text=document,chunk_size=chunk_size,overlap=overlap,)
    chunks: list[dict] = []

    for raw_chunk in raw_chunks:
        body = raw_chunk.text.strip()

        if not body:
            continue

        matched_spans = _pages_for_range(spans=spans,start=raw_chunk.char_start,end=raw_chunk.char_end,)

        if not matched_spans:
            continue

        first_page = matched_spans[0]
        last_page = matched_spans[-1]
        prefix = _context_prefix(section_ref=first_page.section_ref,heading=first_page.heading,)
        chunk_text_with_prefix = prefix + body
        chunk_number = len(chunks) + 1
        chunks.append(
            {
                "chunk_id": f"chunk_{chunk_number:04d}",
                "text": chunk_text_with_prefix,
                "token_count": count_tokens(
                    chunk_text_with_prefix
                ),
                "body_token_count": count_tokens(body),
                "pdf_page_start": first_page.pdf_page,
                "pdf_page_end": last_page.pdf_page,
                "printed_page_start": first_page.printed_page,
                "printed_page_end": last_page.printed_page,
                "section_ref": first_page.section_ref,
                "heading": first_page.heading,
            }
        )

    return chunks


In [13]:
def save_chunks(chunks: list[dict],output_path: Path,) -> None:

    output_path.parent.mkdir(parents=True,exist_ok=True,)

    with output_path.open(mode="w",encoding="utf-8",) as handle:

        for chunk in chunks:
            handle.write(json.dumps(chunk,ensure_ascii=False,) + "\n")

In [14]:
project_root = Path.cwd()

pages_jsonl = project_root / "data" / "processed" / "pages.jsonl"
output_path = project_root / "data" / "processed" / "chunks.jsonl"

chunks = build_chunks(pages_jsonl=pages_jsonl,chunk_size=CHUNK_SIZE,overlap=CHUNK_OVERLAP,)

if not chunks:
    print("Nije napravljen nijedan chunk. Proveri ulazni fajl i podešavanja.")

else:
    save_chunks(chunks, output_path)

    print(f"Sačuvano {len(chunks)} chunk-ova.")
    print(f"Fajl: {output_path.resolve()}")

Sačuvano 344 chunk-ova.
Fajl: /home/jelena/Desktop/faks/masinsko/Student-Question-Answering-from-Course-Materials/data/processed/chunks.jsonl
